# Agentic AI & RAG Engineering — Course Notebook (Weeks 1–3)

**Author:** Narayanan Palani  
**Scope:** Weeks 1 to 3 Complete Implementation Reference  
**Stack:** Python, Ollama (`gpt-oss:20b`), Pydantic v2, AsyncIO, SQLite, FastAPI, Streamlit, Pytest

--- 
## 1. Environment Setup & Dependency Configuration
* **Slide Source:** *Week 1 - Slide 17 ("Setup — Two Minutes"), Week 2 - Slide 26 ("Secrets — Extending the W1 Discipline")*
* **Action:** Install required packages and securely load API credentials using `python-dotenv`.

In [20]:
# Week 1 - Hello LLM
# Purpose: Run a first local LLM request through Ollama.
#
# One-time setup:
#   1. Install Ollama from https://ollama.com using code at cmd prompt: curl -fsSL https://ollama.com/install.sh | sh
#   2. Start the Ollama service in an exclusive command prompt: ollama run gpt-oss:20b
#   3. Pull the local model from Terminal:
#        ollama pull gpt-oss:20b
#
# Ollama runs the model locally, so this inference does not use OpenAI API credits
# or require an OPENAI_API_KEY.

# Install the Ollama Python package in this notebook environment.
# Run this once if the package is not already installed.
%pip install -q ollama

# Import the Ollama Python client.
import ollama

# Send a chat request to the local Ollama model.
response = ollama.chat(
    # Use the same local model throughout this notebook.
    model="gpt-oss:20b",

    # Send only the user message needed for this experiment.
    messages=[
        {
            "role": "user",
            "content": "What is the core benefit of RAG? Answer with exactly one word."
        }
    ]
)

# Print the text generated by the local model.
print(response.message.content)


Note: you may need to restart the kernel to use updated packages.


HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"


Accuracy


In [21]:
# Purpose:
#   1. Verify that the Ollama Python package is available.
#   2. Confirm that the local Ollama service is reachable.
#   3. Confirm that the selected local model is installed.
#
# Frequently Asked Interview Questions:
# Q: How can you programmatically verify that a specific LLM model is available locally before executing prompts?
# A: You can fetch the list of available models using `ollama.list()`. By extracting the model names (`[model.model for model in models.models]`), you can check for the required model and raise a `RuntimeError` if it is missing, preventing downstream pipeline failures.
# Slide Source:
#   Week 1 - Slide 17 ("Setup — Two Minutes")
#   Week 2 - Slide 26 ("Secrets — Extending the W1 Discipline")
#
# Local-model principle:
#   Ollama runs inference on your machine.
#   No OPENAI_API_KEY is required for these exercises.
#   No OpenAI API credits are consumed by local inference calls.
#
# Before running this cell:
#   1. Install Ollama from https://ollama.com
#   2. Start the Ollama service.
#   3. Pull the model from Terminal:
#        ollama pull gpt-oss:20b

import ollama

# Query the local Ollama service for installed models.
# This checks connectivity without generating a model response.
models = ollama.list()

# Extract the local model names.
installed_models = [model.model for model in models.models]

# Stop early with a clear instruction if the required model is missing.
if not any(name == "gpt-oss:20b" for name in installed_models):
    raise RuntimeError(
        f"Local Ollama model 'gpt-oss:20b' was not found. "
        f"Run: ollama pull gpt-oss:20b"
    )

# Confirm that the local environment is ready.
print("Local Ollama environment successfully initialized.")
print(f"Model available: gpt-oss:20b")


HTTP Request: GET http://127.0.0.1:11434/api/tags "HTTP/1.1 200 OK"


Local Ollama environment successfully initialized.
Model available: gpt-oss:20b


--- 
## 2. Week 1: Foundations, Decision Frameworks & Hello LLM
* **Slide Source:** *Week 1 - Slide 06 ("A Working Definition"), Slide 11-16 ("Four Patterns"), Slide 17-18 ("Hello LLM & Lab Step 2")*
* **Action:** Initialize a local Ollama client, run a baseline prompt using `gpt-oss:20b`, and log local inference metrics.

In [22]:
# Local Ollama connectivity check
#
# This replaces a cloud-model/API connectivity check.
# It verifies that:
#   1. The Ollama service is running locally.
#   2. The Python client can communicate with it.
#   3. Local models are visible to the client.
#
# No LLM prompt is generated by this check.

import ollama

# Ask the local Ollama service for its installed model list.
models = ollama.list()

print("Ollama service is reachable.")
print("Installed local models:")

# Display the local models available to this notebook.
for model in models.models:
    print(f"- {model.model}")


HTTP Request: GET http://127.0.0.1:11434/api/tags "HTTP/1.1 200 OK"


Ollama service is reachable.
Installed local models:
- gpt-oss:20b


In [23]:
# Week 1 - Hello LLM / Lab Step 2
#
# This version uses Ollama instead of the local Ollama service.
# The model runs locally, so there is no API key or API credit requirement.
# Frequently Asked Interview Questions:
# Q: How do you measure the computational cost or token generation of a local inference request?
# A: By extracting the `prompt_eval_count` (input tokens) and `eval_count` (completion tokens) attributes directly from the response object using `getattr()`. This allows you to track system resource consumption per request.
# Import the Ollama Python client.
import ollama

# Use one model consistently throughout the notebook.
OLLAMA_MODEL = "gpt-oss:20b"


def run_hello_llm(prompt_text: str) -> str:
    """
    Send a prompt to the local Ollama model and return its response.

    prompt_text:
        The question/instruction sent to the local model.

    Returns:
        The text generated by the local model.
    """

    # Send the prompt to Ollama running on the local machine.
    response = ollama.chat(
        model=OLLAMA_MODEL,
        messages=[
            {
                "role": "user",
                "content": prompt_text
            }
        ],
        options={
            # Keep generation focused and relatively deterministic.
            "temperature": 0.2,
            # Limit generation for this one-word experiment.
            "num_predict": 1000
        }
    )

    # Ollama reports local inference statistics in the response.
    # prompt_eval_count = input tokens evaluated locally.
    # eval_count = output tokens generated locally.
    prompt_tokens = getattr(response, "prompt_eval_count", 0)
    completion_tokens = getattr(response, "eval_count", 0)

    # Calculate total locally processed tokens for visibility.
    total_tokens = prompt_tokens + completion_tokens

    print(
        f"Prompt tokens: {prompt_tokens} | "
        f"Completion tokens: {completion_tokens} | "
        f"Total tokens: {total_tokens}"
    )

    # Extract and return only the generated text.
    return response.message.content


# Keep the prompt short to demonstrate prompt-efficiency principles.
prompt = "What is the core benefit of RAG? Answer with exactly one word."

# Run the local model.
result = run_hello_llm(prompt)

# Display the generated response.
print("\nLLM Response:")
print(result)


HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"


Prompt tokens: 82 | Completion tokens: 285 | Total tokens: 367

LLM Response:
Accuracy


--- 
## 3. Week 2: Typed Contracts (Pydantic) & Async Concurrency Pipeline
* **Slide Source:** *Week 2 - Slide 05-07 ("Pydantic Models"), Slide 09-11 ("Async Basics & httpx"), Slide 15-18 ("Concurrency Patterns: Gather, Batching, Retry"), Slide 23 ("Structured Logging"), Slide 27 ("SQLite Store")*
* **Action:** Build Pydantic schemas, an async client with exponential retry backoff, parallel batch processing via `asyncio.gather`, JSON structured logging, and SQLite persistence.

In [24]:
# Slide Source: Week 2 - Slide 05-07 ("Pydantic Models"), Slide 09-11 ("Async Basics & httpx"),
# Slide 15-18 ("Concurrency Patterns: Gather, Batching, Retry"), Slide 23 ("Structured Logging"), Slide 27 ("SQLite Store")
#
# Action: Build Pydantic schemas, an async Ollama client with exponential retry
# backoff, parallel batch processing via asyncio.gather, JSON structured logging,
# and SQLite persistence.
#
# Ollama replaces the paid cloud API for this notebook. The model runs locally.
# Frequently Asked Interview Questions:
# Q: How do you handle database concurrency and avoid "database locked" errors in SQLite when processing asynchronous LLM batches?
# A: You enable Write-Ahead Logging (`PRAGMA journal_mode=WAL;`), utilize a global `asyncio.Lock()`, and offload the blocking synchronous SQLite write operation to a separate worker thread using `asyncio.to_thread(_sync_save_to_db, record)`.
import asyncio
import json
import logging
import sqlite3
import time
import os
from typing import List
from ollama import AsyncClient
from pydantic import BaseModel, Field

# Use the same local model throughout the notebook.
OLLAMA_MODEL = "gpt-oss:20b"

# Global lock to serialize database writes across concurrent tasks
db_lock = asyncio.Lock()

# Structured Logging Setup
logging.basicConfig(level=logging.INFO, format="%(message)s")


def log_json(event: str, **kwargs):
    log_entry = {"event": event, "timestamp": time.time(), **kwargs}
    logging.info(json.dumps(log_entry))


# Pydantic Schemas
class QuestionRequest(BaseModel):
    id: int
    query: str = Field(..., min_length=3, description="The user query")


class AnswerResponse(BaseModel):
    id: int
    query: str
    answer: str
    status: str = "success"


# --- DB Initialization & Unlock Logic ---

def unlock_and_init_db(db_name="results.db"):
    """
    Checks for locks and initializes the DB.
    If heavily locked by a stale Jupyter thread, it forces an unlock.
    """
    max_retries = 3
    
    for attempt in range(1, max_retries + 1):
        try:
            # Short timeout to quickly check for stale locks
            conn = sqlite3.connect(db_name, timeout=5.0)
            cursor = conn.cursor()

            # Enable Write-Ahead Logging (WAL) mode for drastically better concurrency
            cursor.execute("PRAGMA journal_mode=WAL;")

            cursor.execute(
                """
                CREATE TABLE IF NOT EXISTS run_results (
                    id INTEGER PRIMARY KEY,
                    query TEXT NOT NULL,
                    answer TEXT NOT NULL,
                    created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
                )
                """
            )
            conn.commit()
            conn.close()
            print(f"Database '{db_name}' initialized and verified unlocked.")
            return  # Success, exit function
            
        except sqlite3.OperationalError as e:
            if "locked" in str(e).lower():
                print(f"Database locked (Attempt {attempt}/{max_retries}). Retrying in 2 seconds...")
                time.sleep(2)
            else:
                raise
                
    # If we exhaust retries, forcefully unlock by clearing stale WAL/DB files 
    # (Safe for local notebook experimentation; use caution in production)
    print("Database remained locked. Forcefully clearing lock for notebook environment...")
    for ext in ["", "-wal", "-shm"]:
        filepath = f"{db_name}{ext}"
        if os.path.exists(filepath):
            try:
                os.remove(filepath)
                print(f"Removed stale file: {filepath}")
            except OSError as cleanup_err:
                print(f"Failed to remove {filepath}: {cleanup_err}")
                
    # Re-run initialization now that files are cleared
    print("Re-initializing fresh database...")
    unlock_and_init_db(db_name)


def _sync_save_to_db(record: AnswerResponse):
    """Synchronous SQLite write operation with extended timeout and guaranteed closure."""
    conn = sqlite3.connect("results.db", timeout=20.0)
    
    try:
        cursor = conn.cursor()

        cursor.execute(
            "INSERT INTO run_results (id, query, answer) VALUES (?, ?, ?)",
            (record.id, record.query, record.answer),
        )

        conn.commit()
    finally:
        # Ensure the lock is released when the write is complete
        conn.close()


async def save_to_db(record: AnswerResponse):
    """Thread-safe, non-blocking async wrapper around DB writes."""
    async with db_lock:
        # Offload blocking SQLite I/O to a worker thread
        await asyncio.to_thread(_sync_save_to_db, record)


# Retry Wrapper with Exponential Backoff
async def with_retry(coro_func, *args, max_retries: int = 3, **kwargs):
    for attempt in range(1, max_retries + 1):
        try:
            return await coro_func(*args, **kwargs)
        except Exception as exc:
            log_json(
                "async_retry_attempt",
                attempt=attempt,
                max_retries=max_retries,
                error=str(exc),
            )

            if attempt == max_retries:
                raise

            await asyncio.sleep(2**attempt)


async def process_single_query(
    async_client: AsyncClient, req: QuestionRequest
) -> AnswerResponse:
    async def _call():
        response = await async_client.chat(
            model=OLLAMA_MODEL,
            messages=[{"role": "user", "content": req.query}],
            options={"temperature": 0.3},
        )

        # Extract message content safely
        if hasattr(response, "message"):
            return response.message.content.strip()
        return response["message"]["content"].strip()

    answer_text = await with_retry(_call)

    res = AnswerResponse(id=req.id, query=req.query, answer=answer_text)

    # Safely save to DB asynchronously without lock contention
    await save_to_db(res)

    log_json("query_processed", id=req.id, query=req.query)
    return res


async def batch_process_pipeline(queries: List[QuestionRequest]):
    async_client = AsyncClient()

    tasks = [process_single_query(async_client, q) for q in queries]

    results = await asyncio.gather(*tasks, return_exceptions=True)
    return results


# Driver Code Execution
async def main():
    sample_queries = [
        QuestionRequest(id=1, query="What is Pydantic in Python?"),
        QuestionRequest(
            id=2, query="How does asyncio.gather enable concurrent local model calls?"
        ),
        QuestionRequest(
            id=3, query="Why use SQLite for local execution persistence?"
        ),
    ]

    print("Starting Async Local Ollama Batch Pipeline...")
    batch_results = await batch_process_pipeline(sample_queries)

    for res in batch_results:
        if isinstance(res, Exception):
            print(f"\nBatch item failed: {res}")
        else:
            print(
                f"\nID: {res.id}"
                f"\nQuery: {res.query}"
                f"\nAnswer: {res.answer[:100]}..."
            )


if __name__ == "__main__":
    # 1. Unlock and initialize the DB *before* starting the main async loop
    unlock_and_init_db()
    
    # 2. Proceed into main functionality
    if "get_ipython" in globals():
        import nest_asyncio
        nest_asyncio.apply()
        asyncio.run(main())
    else:
        asyncio.run(main())

Database 'results.db' initialized and verified unlocked.
Starting Async Local Ollama Batch Pipeline...


HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"
HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"
HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"



Batch item failed: UNIQUE constraint failed: run_results.id

Batch item failed: UNIQUE constraint failed: run_results.id

Batch item failed: UNIQUE constraint failed: run_results.id


--- 
## 4. Week 3: FastAPI Web Service & Streaming Endpoint
* **Slide Source:** *Week 3 - Slide 06 ("FastAPI 12-line App"), Slide 08 ("/health Endpoint"), Slide 12-14 ("Streaming via StreamingResponse")*
* **Action:** Write `main.py` containing FastAPI backend supporting a health probe, structured POST endpoint, and streaming token response via SSE / raw stream.

In [25]:
%%writefile main.py
# Slide Source: Week 3 - Slide 06 ("FastAPI 12-line App"), Slide 08 ("/health Endpoint"), Slide 12-14 ("Streaming via StreamingResponse")
#
# Action: Build FastAPI backend supporting a health probe, a structured POST
# endpoint, and a streaming response backed by a local Ollama model.
#
# Run locally with:
#   uvicorn main:app --reload --port 8000
# Frequently Asked Interview Questions:
# Q: How do you implement real-time token streaming for an LLM endpoint in FastAPI?
# A: You create an asynchronous generator (`stream_generator`) that requests a stream from the LLM client (`stream=True`) and yields text chunks as they arrive. This generator is then passed to FastAPI's `StreamingResponse` with a `text/plain` media type.
from fastapi import FastAPI, HTTPException
from fastapi.responses import StreamingResponse
from pydantic import BaseModel
from ollama import AsyncClient

# The local Ollama model used by this backend.
OLLAMA_MODEL = "gpt-oss:20b"

# Create the FastAPI application.
app = FastAPI(
    title="Agentic RAG Engine API",
    version="2.0.0"
)

# Create one asynchronous Ollama client.
# Ollama communicates with the local Ollama service.
async_client = AsyncClient()


class AskRequest(BaseModel):
    # Text supplied by the API caller.
    query: str


@app.get("/health")
async def health_check():
    # Lightweight application health probe.
    # This does not invoke the LLM.
    return {
        "status": "ok",
        "service": "agentic-rag-engine",
        "model": OLLAMA_MODEL
    }


@app.post("/ask")
async def ask_endpoint(request: AskRequest):
    # Reject empty or whitespace-only queries.
    if not request.query.strip():
        raise HTTPException(
            status_code=400,
            detail="Query string cannot be empty."
        )

    # Send the request to the local Ollama model.
    response = await async_client.chat(
        model=OLLAMA_MODEL,
        messages=[
            {"role": "user", "content": request.query}
        ]
    )

    # Return the original query and generated answer as JSON.
    return {
        "query": request.query,
        "answer": response.message.content
    }


async def stream_generator(query: str):
    # Request streaming generation from the local Ollama model.
    response = await async_client.chat(
        model=OLLAMA_MODEL,
        messages=[
            {"role": "user", "content": query}
        ],
        stream=True
    )

    # Yield generated text chunks as they arrive.
    async for chunk in response:
        content = chunk.message.content
        if content:
            yield content


@app.post("/stream")
async def stream_endpoint(request: AskRequest):
    # Return a streaming HTTP response to the caller.
    return StreamingResponse(
        stream_generator(request.query),
        media_type="text/plain"
    )


Overwriting main.py


--- 
## 5. Streamlit Frontend UI
* **Slide Source:** *Week 3 - Slide 16 ("Minimal Streamlit UI")*
* **Action:** Write `app.py` constructing UI consuming the FastAPI streaming response in real-time.

In [26]:
%%writefile app.py
# Slide Source: Week 3 - Slide 16 ("Minimal Streamlit UI")
# Action: Construct UI consuming the FastAPI streaming response in real-time.
# Frequently Asked Interview Questions:
# Q: How do you consume a streaming backend endpoint in a Streamlit interface to display progressively generated text?
# A: You use the `requests.post` method with `stream=True`. By iterating over `response.iter_content()`, you capture incoming chunks and dynamically append them to a Streamlit placeholder using `st.empty().markdown()`.
import streamlit as st
import requests

st.set_page_config(page_title="Agentic RAG Control Center", layout="wide")
st.title("Agentic AI & RAG Interface")

query_input = st.text_input("Enter your request or prompt:", placeholder="Ask something...")

if st.button("Submit Query"):
    if not query_input.strip():
        st.warning("Please enter a valid query.")
    else:
        st.subheader("Streaming Response:")
        response_box = st.empty()
        full_response = ""
        
        try:
            url = "http://localhost:8000/stream"
            with requests.post(url, json={"query": query_input}, stream=True) as response:
                if response.status_code == 200:
                    for chunk in response.iter_content(chunk_size=1024, decode_unicode=True):
                        if chunk:
                            full_response += chunk
                            response_box.markdown(full_response + "▌")
                    response_box.markdown(full_response)
                else:
                    st.error(f"Error {response.status_code}: Unable to reach API.")
        except Exception as e:
            st.error(f"Connection error: {str(e)}")

# Run with command: streamlit run app.py

Overwriting app.py


--- 
## 6. Week 3 (Day 2): Testing, Mocks & API Contract Specification
* **Slide Source:** *Week 3 - Slide 28-30 ("Testing & Mocks"), Slide 33-36 ("API Contracts & ADR 0002")*
* **Action:** Unit test pipeline components with `pytest` & `AsyncMock`, and document Architectural Decision Record (ADR 0002).

In [27]:
# Slide Source: Week 3 - Slide 28-30 ("Testing & Mocks")
#
# Action: Test execution components without making real local-model calls.
# AsyncMock simulates the Ollama AsyncClient response.
#
# The test deliberately avoids inference, so it is free and fast.
# Frequently Asked Interview Questions:
# Q: How do you unit test an asynchronous LLM function without triggering an actual, time-consuming model inference?
# A: You use `unittest.mock.AsyncMock()` to construct a fake asynchronous client and response structure. Using `@pytest.mark.asyncio`, you can `await` the function under test and assert the result, verifying the mock was triggered using `assert_called_once()`.
import pytest
from unittest.mock import AsyncMock
from pydantic import BaseModel


class QuestionRequest(BaseModel):
    id: int
    query: str


class AnswerResponse(BaseModel):
    id: int
    query: str
    answer: str


async def dummy_llm_call(query: str, client) -> str:
    # Call the same method shape used by the production Ollama client.
    response = await client.chat(
        model="gpt-oss:20b",
        messages=[
            {"role": "user", "content": query}
        ]
    )

    # Return the generated message text.
    return response.message.content


@pytest.mark.asyncio
async def test_dummy_llm_call_success():
    # Create a fake asynchronous Ollama client.
    mock_client = AsyncMock()

    # Build a fake response object with the structure expected
    # by dummy_llm_call().
    mock_message = AsyncMock()
    mock_message.content = "Mocked answer payload"

    mock_completion = AsyncMock()
    mock_completion.message = mock_message

    # Configure the fake client's chat() call to return our mock response.
    mock_client.chat.return_value = mock_completion

    # Execute the function under test.
    result = await dummy_llm_call(
        "Test Query",
        mock_client
    )

    # Verify the returned answer.
    assert result == "Mocked answer payload"
    # Retry this result with int value to see unit test failure-
    # assert result == 2

    # Verify that the local-model client was called exactly once.
    mock_client.chat.assert_called_once()

    print("Test passed successfully!")


# Run the asynchronous test directly in the notebook.
await test_dummy_llm_call_success()


Test passed successfully!


### ADR 0002: API Interface Contract Locking

**Context:** Standardizing the interface contract for client applications interacting with the RAG microservice.

**Decision:** All standard interactions will adhere strictly to Pydantic JSON validation schemas.

#### Schema Spec: `/v1/ask`
* **Request (POST):** `{"query": "string (min_length: 3)"}`
* **Response (200 OK):** `{"query": "string", "answer": "string"}`
* **Error Response (400 Bad Request):** `{"detail": "Query string cannot be empty."}`

### Building a Resilient, Asynchronous LLM Pipeline
An architectural breakdown of the Week 2 local Ollama batch processor.

The provided code represents a robust, production-ready pattern for communicating with Local LLMs (Large Language Models) in an asynchronous environment. Instead of making slow, sequential requests, this pipeline processes multiple prompts concurrently while enforcing strict data types and handling failures gracefully.

Here is a breakdown of how the different components work together:

Enforcing Strict Contracts with Pydantic
In agentic workflows, unpredictable data structures lead to broken pipelines. The code uses Pydantic BaseModel classes (Question and Answer) to create strict data contracts. This ensures that every piece of data moving through the pipeline conforms to an expected format, reducing the risk of runtime errors when passing data between the UI, the LLM, and the database.

The Async LLM Engine
The async_ask_ollama function is the core engine. It swaps out the original lab's fake sleep timers for genuine, asynchronous HTTP calls to a local Ollama model (gpt-oss:20b) using ollama.AsyncClient().

To effectively test the resilience of the system, this function also features a deliberate fail_rate parameter. By artificially triggering TransientError exceptions, developers can simulate real-world network hiccups, API rate limits, or locked databases without needing to actually break their local server.

Resilience via Exponential Backoff
AI generation can be resource-intensive, and requests occasionally time out or fail. The ask_llm_with_retry wrapper prevents a single failed request from crashing the entire batch process.

It catches simulated transient failures.

It pauses before trying again using an exponential backoff strategy (waiting 1 second, then 2 seconds, then 4 seconds).

This prevents overwhelming an already struggling server with immediate, repeated requests.

Concurrency Pattern: Batch vs. Stream
The code provides two distinct ways to handle concurrent LLM requests, both utilizing Python's asyncio library:

Strict Ordering (run_batch): Uses asyncio.gather to fire off all questions simultaneously, but waits for the absolute slowest response to finish before returning anything. The output strictly matches the order of the input.

Speed to Insight (run_batch_stream): Uses asyncio.as_completed. As soon as any single LLM call finishes—regardless of its position in the original list—it is immediately processed and printed. This is crucial for user-facing applications (like Streamlit UIs) where you want to show progress immediately rather than making the user stare at a loading screen until the entire batch is done.

In [28]:
# Week 2 - Typed Contracts, Input/Output Pydantic Validation Engine & Async Pipeline
# Frequently Asked Interview Questions:
# Q: How can Pydantic be used to enforce anti-hallucination guardrails on an LLM's output?
# A: You define a `BaseModel` (like `LLMResponseSchema`) representing the target output. During runtime parsing, you compare the LLM's cited references (e.g., `referenced_txn_ids`) against a known, pre-validated set of source IDs. If a hallucinated ID is cited, you raise a `ValueError` to reject the response.
import asyncio
import random
import json
import re
import traceback
from pydantic import BaseModel, ConfigDict, Field, field_validator, ValidationError
from typing import Optional, List
from ollama import AsyncClient


# ─── 1. RAG Input Data Validation Engine ────────────────────────────────

class TransactionRecord(BaseModel):
    """Enforces strict structural rules on incoming raw RAG data."""
    model_config = ConfigDict(extra='forbid') # Rejects unrecognized keys
    
    transaction_id: str
    customer_id: str
    amount: float
    currency: str
    store_location: Optional[str] = None
    timestamp: str
    payment_method: Optional[str] = None
    status: str

    @field_validator('amount')
    @classmethod
    def validate_amount(cls, value: float) -> float:
        if value < 0:
            raise ValueError(f"Negative amount detected: {value}")
        return value

    @field_validator('timestamp')
    @classmethod
    def validate_timestamp(cls, value: str) -> str:
        if value == "Bad-Time":
            raise ValueError("Corrupted timestamp format ('Bad-Time')")
        return value


def process_rag_context(raw_json_str: str) -> tuple[str, set[str]]:
    """
    Filters raw context into ACCEPTED, REJECTED, and IGNORED buckets.
    Also returns a set of valid Transaction IDs for anti-hallucination checks.
    """
    try:
        raw_records = json.loads(raw_json_str)
    except json.JSONDecodeError as e:
        print(f"⚠️ [Input Data Error] Failed to parse JSON: {e}")
        return "{}", set()

    categorized_data = {"ACCEPTED": [], "REJECTED": [], "IGNORED": []}
    valid_txn_ids = set()

    for record in raw_records:
        try:
            valid_txn = TransactionRecord(**record)
            if valid_txn.status.lower() == "unsuccessful":
                categorized_data["IGNORED"].append(valid_txn.model_dump())
            else:
                categorized_data["ACCEPTED"].append(valid_txn.model_dump())
                valid_txn_ids.add(valid_txn.transaction_id)
        except ValidationError as e:
            categorized_data["REJECTED"].append({
                "original_record": record,
                "validation_errors": [err["msg"] for err in e.errors()]
            })
            
    return json.dumps(categorized_data, indent=2), valid_txn_ids


# ─── 2. Response Validation Contract (LLM Output Guardrail) ──────────────

class LLMResponseSchema(BaseModel):
    """
    Target structure expected back from the LLM response.
    Includes built-in validation rules for grounding and formatting.
    """
    answer_summary: str = Field(description="Direct, non-empty answer to the query")
    referenced_txn_ids: List[str] = Field(default_factory=list, description="IDs cited in response")
    confidence_score: float = Field(ge=0.0, le=1.0, description="Confidence rating between 0 and 1")

    @field_validator('answer_summary')
    @classmethod
    def check_non_empty(cls, value: str) -> str:
        if not value.strip():
            raise ValueError("LLM generated an empty answer summary.")
        return value.strip()


# ─── 3. Exclusive Runtime Response Validation Engine ────────────────────

def validate_and_parse_llm_response(
    raw_response_text: str, 
    valid_ids_context: Optional[set[str]] = None
) -> LLMResponseSchema:
    """
    Exclusive Runtime Validation Step:
    1. Extracts JSON payload from potential raw markdown wrappers (```json ... ```).
    2. Validates against `LLMResponseSchema`.
    3. Performs Anti-Hallucination checks against valid transaction IDs.
    """
    # Step A: Clean Markdown wrapping if present
    clean_text = raw_response_text.strip()
    json_match = re.search(r'```(?:json)?\s*(\{.*?\})\s*```', clean_text, re.DOTALL)
    if json_match:
        clean_text = json_match.group(1)

    # Step B: Parse JSON string into dict
    try:
        data_dict = json.loads(clean_text)
    except json.JSONDecodeError:
        # Fallback if LLM provided plain text instead of JSON schema
        data_dict = {
            "answer_summary": raw_response_text.strip(),
            "referenced_txn_ids": [],
            "confidence_score": 0.8  # Default fallback score
        }

    # Step C: Pydantic Structural Validation
    validated_response = LLMResponseSchema(**data_dict)

    # Step D: Grounding / Anti-Hallucination Guardrail Validation
    if valid_ids_context:
        for txn_id in validated_response.referenced_txn_ids:
            if txn_id not in valid_ids_context:
                raise ValueError(
                    f"Hallucination Detected! Transaction ID '{txn_id}' cited by LLM does not exist in context."
                )

    return validated_response


# ─── 4. Pipeline Typed Models & LLM Integration ────────────────────────

class Question(BaseModel):
    text: str
    context: Optional[str] = None
    valid_txn_ids: Optional[set[str]] = None

class Answer(BaseModel):
    question_text: str
    text: str
    confidence: Optional[float] = None
    is_error: bool = False
    error_type: Optional[str] = None

class TransientError(Exception):
    """Simulates transient API failures."""
    pass


async def async_ask_ollama(question: Question, fail_rate: float = 0.0) -> Answer:
    if random.random() < fail_rate:
        await asyncio.sleep(0.1)
        raise TransientError("Simulated transient failure")
    
    system_prompt = (
        "You are a strict data assistant. Respond to the user request ONLY using a JSON object with this exact structure:\n"
        "{\n"
        '  "answer_summary": "Your detailed answer string",\n'
        '  "referenced_txn_ids": ["TXN-xxxx"],\n'
        '  "confidence_score": 0.95\n'
        "}\n"
    )
    
    messages = [{"role": "system", "content": system_prompt}]
    if question.context:
        messages.append({
            "role": "system", 
            "content": f"Context Data: {question.context}"
        })
        
    messages.append({"role": "user", "content": question.text})

    # Call Ollama API
    response = await AsyncClient().chat(
        model="gpt-oss:20b",
        messages=messages
    )
    raw_content = response['message']['content']

    # ─── EXCLUSIVE RESPONSE VALIDATION EXECUTION ───
    try:
        parsed_output = validate_and_parse_llm_response(
            raw_response_text=raw_content, 
            valid_ids_context=question.valid_txn_ids
        )
        return Answer(
            question_text=question.text,
            text=parsed_output.answer_summary,
            confidence=parsed_output.confidence_score,
            is_error=False
        )
    except (ValidationError, ValueError) as resp_err:
        # Gracefully handle and output response-level validation failures
        err_details = str(resp_err).replace('\n', ' ')
        return Answer(
            question_text=question.text,
            text=f"Raw LLM Output: {raw_content[:60]}...",
            is_error=True,
            error_type=f"Response Validation Failed ({err_details})"
        )


# ─── 5. Async Execution & Error Loggers ─────────────────────────────────

async def ask_llm_with_retry(question: Question, fail_rate: float = 0.0, max_attempts: int = 3) -> Answer:
    for attempt in range(max_attempts):
        try:
            return await async_ask_ollama(question, fail_rate=fail_rate)
            
        except TransientError:
            if attempt == max_attempts - 1:
                return Answer(
                    question_text=question.text,
                    text="Max retries reached due to transient network failures.",
                    is_error=True,
                    error_type="Network Transient Fail"
                )
            backoff = 2 ** attempt  
            print(f"    [Retry] Attempt {attempt + 1} failed. Retrying in {backoff}s...")
            await asyncio.sleep(backoff)
            
        except Exception as e:
            # System errors (e.g., Ollama server offline / Model missing)
            error_msg = f"{type(e).__name__}: {str(e)}"
            return Answer(
                question_text=question.text, 
                text=f"Execution aborted: {error_msg}", 
                is_error=True,
                error_type="System API Exception"
            )

async def run_batch(questions: list[Question], fail_rate: float = 0.0) -> list[Answer]:
    """All-or-nothing: returns when slowest call finishes, in input order."""
    tasks = [ask_llm_with_retry(q, fail_rate=fail_rate) for q in questions]
    
    # asyncio.gather runs them all at once and returns the results in the original order
    # Because ask_llm_with_retry catches errors and returns Answer(is_error=True), 
    # gather won't crash if a single task fails!
    return await asyncio.gather(*tasks)
    
async def run_batch_stream(questions: list[Question], fail_rate: float = 0.0) -> list[Answer]:
    tasks = [ask_llm_with_retry(q, fail_rate=fail_rate) for q in questions]
    results: list[Answer] = []

    for coro in asyncio.as_completed(tasks):
        try:
            ans = await coro
            print(f"❓ Question: {ans.question_text}")
            
            if ans.is_error:
                print(f"❌ STATUS: FAILED | Type: {ans.error_type}")
                print(f"   Log Detail: {ans.text}\n")
            else:
                print(f"✅ STATUS: PASSED | Confidence: {ans.confidence}")
                print(f"   Answer:   {ans.text}\n") 
                
            print("-" * 65)  
            results.append(ans)
        except Exception as e:
            print(f"💥 [Critical Unhandled Loop Error]: {e}")
            traceback.print_exc()
        
    return results


# ─── 6. Execution Block ─────────────────────────────────────────────────

mock_json_content = """
[
    {
        "transaction_id": "TXN-1001", "customer_id": "CUST-100", "amount": 150.0,
        "currency": "USD", "store_location": "NY", "timestamp": "2023-10-01T12:00:00Z",
        "payment_method": "Credit", "status": "successful"
    },
    {
        "transaction_id": "TXN-1002", "customer_id": "CUST-200", "amount": -25.5,
        "currency": "EUR", "store_location": "LDN", "timestamp": "2023-10-01T12:05:00Z",
        "payment_method": "Debit", "status": "successful"
    }
]
"""

# Step 1: Execute Request/Input Validation Engine
filtered_rag_data, valid_txn_ids = process_rag_context(mock_json_content)

sample_questions = [
    Question(text=t, context=filtered_rag_data, valid_txn_ids=valid_txn_ids) 
    for t in [
        "What are the details of transaction TXN-1001?",
        "Which customer made a transaction with a negative amount?"
    ]
]

print("\nStarting Async Pipeline (Input Validation + Response Validation Enabled)\n")

async def main():
    answers = await run_batch_stream(sample_questions, fail_rate=0.0)
    passed = sum(1 for a in answers if not a.is_error)
    failed = sum(1 for a in answers if a.is_error)
    print(f"\nExecution Finished. Passed Validation: {passed} | Failed Validation: {failed}")

# In Jupyter Notebook: 
await main()
# In standard Python script: asyncio.run(main())


Starting Async Pipeline (Input Validation + Response Validation Enabled)



HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"


❓ Question: What are the details of transaction TXN-1001?
✅ STATUS: PASSED | Confidence: 0.95
   Answer:   Transaction TXN-1001: Customer CUST-100 spent $150.00 USD at the NY store location on 2023-10-01T12:00:00Z using Credit card, transaction status is successful.

-----------------------------------------------------------------


HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"


❓ Question: Which customer made a transaction with a negative amount?
❌ STATUS: FAILED | Type: Response Validation Failed (Hallucination Detected! Transaction ID 'TXN-1002' cited by LLM does not exist in context.)
   Log Detail: Raw LLM Output: {"answer_summary":"Customer CUST-200 made a transaction with...

-----------------------------------------------------------------

Execution Finished. Passed Validation: 1 | Failed Validation: 1


In [29]:
# Run the batch execution
answersBatch = await run_batch(sample_questions, fail_rate=0.0)

print(f"\nSuccessfully returned {len(answersBatch)} real LLM answers.\n")

# Loop through and print questions with their corresponding validation statuses
for answer in answersBatch:
    print(f"❓ Question: {answer.question_text}")
    
    if answer.is_error:
        print(f"❌ STATUS: FAILED | Type: {answer.error_type}")
        print(f"   Log Detail: {answer.text.strip()}\n")
    else:
        conf_str = f"{answer.confidence:.2f}" if answer.confidence is not None else "N/A"
        print(f"✅ STATUS: PASSED | Confidence: {conf_str}")
        print(f"   Answer:   {answer.text.strip()}\n")
        
    print("-" * 65)

HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"
HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"



Successfully returned 2 real LLM answers.

❓ Question: What are the details of transaction TXN-1001?
✅ STATUS: PASSED | Confidence: 0.95
   Answer:   Transaction TXN-1001 details:
• Transaction ID: TXN-1001
• Customer ID: CUST-100
• Amount: 150.0 USD
• Currency: USD
• Store Location: NY (New York)
• Timestamp: 2023-10-01T12:00:00Z
• Payment Method: Credit
• Status: successful

-----------------------------------------------------------------
❓ Question: Which customer made a transaction with a negative amount?
❌ STATUS: FAILED | Type: Response Validation Failed (Hallucination Detected! Transaction ID 'TXN-1002' cited by LLM does not exist in context.)
   Log Detail: Raw LLM Output: {
  "answer_summary": "The customer with ID \"CUST-200\" mad...

-----------------------------------------------------------------
